# 3.7 Nested Conditionals

## Learning Objective

By the end of this lesson you can write a nested conditional in Python that handles **every** combination of its conditions, and prove that it does by counting leaves.

## Success Criteria

- Given two conditions, you can state how many outcomes exist *before* writing any code.
- Your conditional produces output for every combination of inputs, not just the one you were picturing.
- You can point at any input and name which branch it lands in.

> **The whole lesson in one sentence:** an `if` with no `else` is a question with a missing answer, and Python will never tell you it's missing.


## 1. LxD Cycle Process

### Empathize

We found the misconception inside our own course material. The 3.7 lesson already in this repo (`_notebooks/CSP/big-ideas/big-idea-3/3-07-nested-conditionals/`) opens with this:

```javascript
if (time < 3 && score > 1000) {
    lives = lives + 3;
    level = level + 1;
}
```

Run it with `time = 4.2` and nothing happens. No error, no warning, no output. The lesson even flags it a moment later — *"Problem: What if the conditions aren't met?"* — which tells us the person writing it already knew.

So the gap isn't knowing that branches can be missing. It's **noticing that one is missing in your own code**, before somebody else finds it. That is harder in Python than almost anywhere else, because a Python program with a missing branch runs to completion and exits with status 0. Success and "did nothing" look identical from the outside.

### Define

- **POV:** A CSP student needs a way to check that a conditional covers every case, because a missing branch produces no error message — the program succeeds and does nothing, which is indistinguishable from working code.
- **Learning goal:** Students will write nested conditionals in Python that cover every combination of two conditions, and verify that coverage by counting leaves against 2<sup>n</sup>.

### Ideate

- **HMW:** How might we make a missing branch visible *before* the code runs, when the bug itself produces no error?
- **Activity chosen — Count the leaves.** Count the outcomes on paper first (2 conditions, so 4), then count the leaves in the code. If the two numbers disagree, you have a bug — and you found it without running anything.

### Prototype & Test

Built as the notebook you're reading, taught to our own team first. What changed as a result is written up in section 5.


## 2. Lesson Plan

### Tech Talk (3 minutes)

A nested conditional is a conditional inside a conditional. The outer answer decides which question you ask next.

In Python the nesting **is** the indentation. You can read the depth without reading a single condition:

```
if outer:
    if inner:        <- two levels in
        ...
    else:
        ...
else:
    if inner:
        ...
    else:
        ...
```

**The rule:** `n` conditions produce 2<sup>n</sup> possible combinations of inputs.

| Conditions | Possible outcomes |
|---:|---:|
| 1 | 2 |
| 2 | 4 |
| 3 | 8 |

A **leaf** is a branch with nothing conditional inside it — a place the program actually stops and does something. Your code needs one leaf per outcome, or a reason it has fewer.

**Why counting matters here specifically.** In most languages a missing case is just wrong. In Python it's wrong *and silent*: `if x:` with no `else`, and `x` false, prints nothing, raises nothing, and exits 0.

**The thing that trips people up.** `and` collapses four outcomes into two. `if a and b:` gives you "both true" and "everything else" — it cannot tell "only `a`" from "only `b`" from "neither." When those three need different answers, you have to nest.


### Example A — Simple: the branch that isn't there

Our capstone is the Friends of Poway Seniors site, which has a volunteer signup form. A volunteer is cleared to work an event when they are 18 or older **and** they've finished training.

Here's that check, written the way most people write it the first time. Run it.


In [ ]:
# CODE_RUNNER: 3.7A - The branch that isn't there: run it, what prints?
age = 16
training_complete = True

if age >= 18 and training_complete:
    print("Cleared to volunteer.")


That cell produced **no output**. It also produced **no error**. Skimming past it, you'd assume it worked.

Two conditions means 4 possible inputs. Count the leaves in that code: there is **1**. The other three cases fall off the end of the program and disappear.

Here's the smallest possible fix — one `else`:


In [ ]:
# CODE_RUNNER: 3.7A - One else: now every input gets an answer
age = 16
training_complete = True

if age >= 18 and training_complete:
    print("Cleared to volunteer.")
else:
    print("Not cleared.")


Now every input produces output. 4 outcomes, 2 leaves, nothing vanishes silently.

But `"Not cleared."` is a bad answer. A 16-year-old who *finished* training and a 40-year-old who *skipped* it get the identical message, and neither one learns what to do next. `and` threw away the information about **which** condition failed.

To answer differently in each case, you have to ask the second question *inside* the answer to the first.


### Example B — Intermediate: all four leaves

This one comes from the scam-defense trainer we're building. The trainer looks at a message and decides what to tell the user. Two things matter:

- does the message **ask for money** — gift cards, wire transfer, "can you send me"
- is the sender a **known contact**

All four combinations need genuinely different answers, including the one people get wrong: when a *known contact* asks for money, that usually means their account was taken over.


In [ ]:
# CODE_RUNNER: 3.7B - Scam triage: change the values until all four leaves print
asks_for_money = True
known_contact = True

print(f"asks_for_money={asks_for_money}, known_contact={known_contact}")
print("Advice:")

# 2 conditions -> 4 outcomes -> 4 leaves
if asks_for_money:
    if known_contact:
        # leaf 1
        print("Do not send anything yet. Accounts get taken over.")
        print("Call this person on a number you already have and ask them directly.")
    else:
        # leaf 2
        print("Treat this as a scam. Do not reply and do not send money.")
        print("Block the sender and report the message.")
else:
    if known_contact:
        # leaf 3
        print("Normal message from someone you know. Nothing to do.")
    else:
        # leaf 4
        print("Unknown sender, but no money request. Most likely spam - ignore it.")


#### Count the leaves

2 conditions gives 2<sup>2</sup> = **4 outcomes**. The code has **4 leaves**. The numbers match, so the coverage is complete.

| `asks_for_money` | `known_contact` | Leaf | Response |
|---|---|---:|---|
| `True` | `True` | 1 | Verify by phone — the account may be compromised |
| `True` | `False` | 2 | Treat as a scam, block and report |
| `False` | `True` | 3 | Normal message |
| `False` | `False` | 4 | Probably spam, ignore |

Look at rows 1 and 2. Both have `asks_for_money = True`, and they get opposite advice.

Now imagine writing this as `if asks_for_money and not known_contact:` instead. That merges rows 1, 3 and 4 into one "everything else" — and row 1, the most dangerous line in the table, a scammer operating a hacked account the user trusts, gets handled as *"nothing to do."*

**Try it:** change the two values at the top of the cell and re-run until you've seen all four leaves print.


### Example C — Complex: three conditions, and why four leaves is enough

Multiplayer bingo. Before a player joins a room we check three things: the room exists, it has space, and the player typed a display name.

Three conditions means 2<sup>3</sup> = **8 combinations**. The code below has only **4 leaves** — and that is correct, not a bug. Read it, then read why.


In [ ]:
# CODE_RUNNER: 3.7C - Bingo room guard: 8 combinations, 4 leaves
room_exists = True
room_has_space = True
display_name = ""

# 3 conditions -> 8 outcomes -> 4 leaves (collapses explained below)
if not room_exists:
    # leaf 1 - absorbs 4 combinations at once.
    # If the room does not exist, space and name describe nothing.
    print("No room with that code. Check the code and try again.")
else:
    if not room_has_space:
        # leaf 2 - absorbs 2 combinations (the name is irrelevant once the room is full)
        print("That room is full. Ask the host to start another table.")
    else:
        if display_name == "":
            # leaf 3
            print("Enter a display name so the other players can see you.")
        else:
            # leaf 4
            print(f"Joining the room as {display_name}. Good luck!")


#### Why 4 leaves can cover 8 combinations

| Leaf | Combinations covered | Why they collapse |
|---|---:|---|
| 1 — no such room | 4 | If the room doesn't exist, `room_has_space` and `display_name` describe nothing that exists |
| 2 — room full | 2 | Once the room is full, the display name can't change the answer |
| 3 — no display name | 1 | |
| 4 — joined | 1 | |
| **Total** | **8** | |

Fewer leaves than 2<sup>n</sup> is fine **when you can say which combinations collapsed and why**. That sentence is the entire difference between a guard clause and a missing branch. If you can't name them, you didn't collapse a case — you forgot one.

This shape, checking the thing that makes the other checks meaningless *first*, is called a **guard**. Order matters: move the room check below the space check and you're asking whether a room that doesn't exist has room in it.


### The same logic in JavaScript

Same four leaves, same order — only the punctuation changes. Braces instead of indentation, `else if` instead of `elif`, `&&` instead of `and`.


In [ ]:
%%js
// CODE_RUNNER: 3.7B JS - The same four leaves in JavaScript
let asksForMoney = true;
let knownContact = true;

console.log(`asksForMoney=${asksForMoney}, knownContact=${knownContact}`);

// 2 conditions -> 4 outcomes -> 4 leaves
if (asksForMoney) {
    if (knownContact) {
        console.log("Do not send anything yet. Call them on a number you already have.");
    } else {
        console.log("Treat this as a scam. Block and report.");
    }
} else {
    if (knownContact) {
        console.log("Normal message from someone you know.");
    } else {
        console.log("Unknown sender, no money request. Likely spam.");
    }
}


**The one difference worth remembering.** In JavaScript the indentation is a courtesy — the braces define the nesting, and the code behaves identically if you delete every space. In Python the indentation *is* the nesting, so a misplaced `else` doesn't just look wrong, it attaches itself to a different `if` and changes what the program means.

Everything else on this page — counting outcomes, counting leaves, collapsing with a guard — is language-independent. It's true in JavaScript, in Java, and in the pseudocode on the AP exam.


## 3. Hacks & Practice Tasks

### Prepare your submission notebook

1. Create a new notebook in your portfolio under `_notebooks/homework`.
2. Add one markdown cell at the top with the frontmatter below.
3. Add one code cell for the Popcorn Hack and one for the Homework Hack.
4. Run every cell and check the output before you submit.

```
---
layout: post
title: 3.7 Nested Conditionals HW
categories: [Python]
lesson_language: Python
lesson_topic: Nested Conditionals
lesson_part: interactive
lesson_type: lesson
permalink: /python/nested-conditionals-hw
author: githubID
---
```

### Submission rules — read first

- Submit **working** Python. If the cell raises an error, that part scores zero.
- **Every leaf must `print()` something.** A branch that does nothing is the exact bug we're grading against.
- Use `if` / `elif` / `else` only. No `match`, no dictionaries of functions, no one-line ternaries.
- Keep the leaf-count comment. It's worth points on its own.


### Popcorn Hack — in class, 2 minutes

The senior center runs an outdoor walking group. It goes ahead when it's warm enough **and** it isn't raining.

**Run the cell below first, without changing anything.** Then answer in chat:

> What does this program print, and is that an error or a bug?

Then fix it so that all four combinations produce output, and paste your fixed code in chat.


In [ ]:
# CODE_RUNNER: 3.7 Popcorn - Run it first: what prints, and is that a bug or an error?
temperature = 68
is_raining = True

if temperature > 75 and not is_raining:
    print("Walking group is on. Meet at the front entrance.")


### Homework Hack

The senior center charges $25 for a class, and members get in free. This code was meant to handle both, and it doesn't:

```python
balance = 40
is_member = False

if balance >= 25:
    if is_member:
        print("Registered for the class.")
```

**Your task:**

1. **Count the outcomes.** Two conditions — how many combinations? Write the number down.
2. **Rewrite it** as a nested conditional with one leaf per outcome. Every leaf prints something a real person could act on: `"Registered."`, `"You're a member, no charge."`, `"You need $8 more."`, and so on.
3. **Add a leaf-count comment** on the line above your outer `if`, in exactly this form:

   ```python
   # 2 conditions -> 4 outcomes -> 4 leaves
   ```

4. **Add a markdown cell** below your code containing a truth table: one row per combination, with the leaf number it reaches.
5. **Test all four** by changing `balance` and `is_member`, and confirm each one prints.

**Stretch, not graded:** add a third condition, `class_is_full`, as a guard at the top. In a comment, say how many of the 8 combinations your guard leaf absorbs and why.


## 4. Grading Plan — 1 point total

### Rubric

**0.2 — Popcorn Hack**

Student answered what the original program prints (nothing — and that it's a bug, not an error) and submitted runnable code covering all four combinations.

**0.8 — Homework Hack**

| Points | Criterion | What we look for |
|---:|---|---|
| 0.3 | Complete coverage | All four combinations of `balance` and `is_member` produce output. Nothing falls off the end of the program. |
| 0.2 | Correct nesting | The second condition is checked *inside* a branch of the first, and the indentation puts each `else` with the `if` it belongs to. |
| 0.2 | Leaf-count comment | Present, the number is right, and it matches the number of leaves actually in the code. |
| 0.1 | Truth table | Four rows, each mapped to the leaf it reaches, and the mapping is correct. |

### Quick validation checklist

- **Present:** a leaf-count comment above the outer `if`.
- **Present:** four distinct `print()` calls, one per leaf.
- **Present:** a markdown truth table with four rows.
- **Absent:** any combination of inputs that produces no output.
- **Absent:** `match`, ternaries, or any construct other than `if` / `elif` / `else`.
- **Runs:** the cell executes without raising.


## 5. Lesson Revisions & Feedback Evidence

**Feedback Received:** Reviewing the draft against the other lessons on the Python lessons page, we found ours would never appear there. It had been built with the big-idea lesson frontmatter, but the page's grid only lists lessons carrying `categories: [Python, ...]`. A lesson nobody can find from the unit page is a lesson nobody does.

**Revision Made:** Moved the notebook in with the other Python lesson sources and rewrote its frontmatter to match the 3.5 Boolean Expressions lesson — `categories`, `lesson_topic`, and `lesson_part: interactive`. It now appears as a card alongside the rest of Big Idea 3, which is where students will actually look for it.


## Summary

- A **nested conditional** is a conditional inside a conditional. The outer answer decides which question you ask next.
- **n conditions produce 2<sup>n</sup> outcomes.** Count them before you write any code.
- A **leaf** is a branch that does something and asks nothing further. Leaves should equal outcomes.
- Fewer leaves is fine **only if you can name which combinations collapsed and why** — that's a guard. If you can't name them, it's a missing branch.
- **`and` collapses four outcomes into two.** Use it when you genuinely only need "both" versus "everything else." Nest when the different failures need different answers.
- In Python a missing branch is **silent**: no output, no error, exit status 0. The leaf count is your only warning.
